# Chapitre 3 — Ajuster un modèle · Corrigé

*Maths pour l'IA, pas à pas*, Belk Siri — notebook compagnon.

[Ouvrir dans Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/fr/ch03_corrige.ipynb) · Page compagnon : https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/

Chaque réponse commence par le raisonnement sur papier, puis montre son équivalent en Python.

**Perte utilisée** : l'erreur quadratique moyenne (équation 3.2) sur les logements A à F, sauf mention contraire.

**Sommaire.** Données · Les calculs des leçons 3.1 à 3.6 · Exercices 3.1 à 3.6

*La cellule de la leçon 3.4 et celle de l'exercice 3.4 f) enchaînent des centaines de milliers de pas : comptez quelques secondes chacune.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Couleurs du livre : une couleur = un rôle (observé, prédit, écart, droite du collègue)
BLEU, ORANGE, ECART, VERT = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Données

L'identifiant D0.1 est celui du livre et du fichier CSV.

In [ ]:
# Données du chapitre 3, recopiées du fichier CSV du livre (ne pas modifier)

# D0.1 — les six logements de l'ouverture, utilisés pour l'ajustement
logements = pd.DataFrame({
    'logement':   ['A', 'B', 'C', 'D', 'E', 'F'],
    'surface_m2': [32, 45, 54, 70, 85, 98],
    'prix_kEUR':  [118, 172, 185, 245, 279, 335],
})
x = logements['surface_m2'].to_numpy(dtype=float)
y = logements['prix_kEUR'].to_numpy(dtype=float)

# D3.1 — quatre ventes nouvelles, utilisées seulement pour évaluer (leçon 3.6)
ventes_nouvelles = pd.DataFrame({
    'vente':      ['V1', 'V2', 'V3', 'V4'],
    'surface_m2': [38, 58, 76, 92],
    'prix_kEUR':  [132, 215, 254, 322],
})
xv = ventes_nouvelles['surface_m2'].to_numpy(dtype=float)
yv = ventes_nouvelles['prix_kEUR'].to_numpy(dtype=float)

# Logement G (ouverture) : exemple de calcul, jamais utilisé pour ajuster
x_G, y_G = 62, 209

## Les calculs des leçons, en Python

Ces cellules refont les calculs des leçons 3.1 à 3.6.

In [ ]:
# Leçon 3.1 — deux droites, deux mesures d'erreur (équations 3.1 et 3.2)
def eam(a, b):
    return np.mean(np.abs(y - (a * x + b)))

def eqm(a, b):
    return np.mean((y - (a * x + b)) ** 2)

for nom, a, b in [("ouverture", 3, 30), ('collègue', 3.3, 14)]:
    print(f'{nom:10s} EAM = {eam(a, b):.2f} k€   EQM = {eqm(a, b):.2f} k€²   REQM = {np.sqrt(eqm(a, b)):.2f} k€')

In [ ]:
# Leçon 3.2 — la courbe de perte, b fixé à 30 k€ (équation 3.3)
pentes = np.round(np.arange(2.5, 3.45, 0.1), 1)
print(pd.Series([round(eqm(a, 30), 2) for a in pentes], index=pentes))
A = np.linspace(2.4, 3.6, 241)
plt.figure(figsize=(6, 3))
plt.plot(A, [eqm(a, 30) for a in A], color=BLEU)
plt.scatter(pentes, [eqm(a, 30) for a in pentes], color='black', zorder=3)
plt.xlabel('Pente a (k€ par m²)'); plt.ylabel('Perte L(a) (k€²)'); plt.grid(alpha=0.4)
plt.show()

In [ ]:
# Leçon 3.3 — sécantes, dérivée (équation 3.4) et deux mises à jour (équation 3.5)
def derivee_a(a, b=30):
    e = y - (a * x + b)
    return -2 / len(x) * np.sum(x * e)

for h in (0.1, 0.01, 0.001):
    print(f'sécante h = {h} : {(eqm(2.9 + h, 30) - eqm(2.9, 30)) / h:.1f}')
print('dérivée exacte en 2,9 :', round(derivee_a(2.9), 1))

a, eta = 2.5, 0.00006
for etape in range(3):
    print(f'étape {etape} : a = {a}, L = {eqm(a, 30):.2f}, L\'(a) = {derivee_a(a):.2f}')
    a = round(a - eta * derivee_a(a), 3)    # arrondi au millième, comme dans le livre

In [ ]:
# Leçon 3.4 — le gradient (équation 3.6) et un pas sur les deux coefficients (équation 3.7)
def gradient(a, b):
    e = y - (a * x + b)
    return np.array([-2 / len(x) * np.sum(x * e), -2 / len(x) * np.sum(e)])

g = gradient(2.5, 30)
print('gradient en (2,5 ; 30) :', g.round(2))
print('après un pas :', (np.array([2.5, 30]) - 0.00006 * g).round(4))

a, b = 2.5, 30.0
for k in range(1, 100001):
    a, b = np.array([a, b]) - 0.00006 * gradient(a, b)
    if k in (10, 1000, 100000):
        print(f'après {k} pas : a = {a:.4f}, b = {b:.4f}, L = {eqm(a, b):.2f}')
a_mc, b_mc = np.polyfit(x, y, 1)
print(f'fond de la vallée : a = {a_mc:.3f}, b = {b_mc:.2f}, L = {eqm(a_mc, b_mc):.2f}')
print(f'G, exemple de calcul : {a_mc * x_G + b_mc:.0f} k€ prédits pour {y_G} k€ observés')

In [ ]:
# Leçon 3.5 — deux descentes depuis les mêmes prédictions (équation 3.8)
mu, sigma = x.mean(), x.std()          # std() divise par n, comme au chapitre 1
z = (x - mu) / sigma
print(f'mu = {mu:.0f} m², sigma = {sigma:.3f} m², z = {z.round(2)}')

def descente_m2(eta, nb_pas, a=2.5, b=30.0):
    for _ in range(nb_pas):
        a, b = np.array([a, b]) - eta * gradient(a, b)
    return a, b

def descente_std(eta, nb_pas, a=2.5, b=30.0):
    w, c = a * sigma, b + a * mu        # équation 2.7 : mêmes prédictions au départ
    for _ in range(nb_pas):
        e = y - (w * z + c)
        w, c = w + eta * 2 / len(x) * np.sum(z * e), c + eta * 2 / len(x) * np.sum(e)
    return w / sigma, c - w / sigma * mu  # équation 3.8 : retour en (a ; b)

print('pas      | m², η = 0,000 06           | standardisées, η = 0,3')
a, b = 2.5, 30.0
fait = 0
for k in (0, 1, 5, 10, 1000, 100000):
    a, b = descente_m2(0.00006, k - fait, a, b); fait = k
    a_s, b_s = descente_std(0.3, min(k, 1000))
    print(f'{k:>8} | a = {a:.3f} b = {b:6.2f} L = {eqm(a, b):8.2f} | '
          f'a = {a_s:.3f} b = {b_s:6.2f} L = {eqm(a_s, b_s):8.2f}')

# Taux limites : au-delà, la descente diverge (environ 0,000 22 en m², 1 en standardisé)
for eta in (0.000216, 0.000217):
    print(f'm², η = {eta} : perte après 400 pas = {eqm(*descente_m2(eta, 400)):.4g} k€²')
for eta in (0.99, 1.01):
    print(f'standardisées, η = {eta} : perte après 400 pas = {eqm(*descente_std(eta, 400)):.4g} k€²')

In [ ]:
# Leçon 3.6 — droite des moindres carrés (équation 3.9) et R² (équation 3.10)
cov = np.mean((x - x.mean()) * (y - y.mean()))
a_mc = cov / x.var()
b_mc = y.mean() - a_mc * x.mean()
print(f'cov = {cov:.1f} m²·k€, var = {x.var():.2f} (m²)², a = {a_mc:.3f} k€/m², b = {b_mc:.2f} k€')

def r2_lecon(observes, predits, reference):
    """R² = 1 − EQM de la règle / EQM de la prédiction constante « reference »."""
    return 1 - np.mean((observes - predits) ** 2) / np.mean((observes - reference) ** 2)

y_moy = y.mean()                      # prix moyen de A à F : la référence du R² sur A à F
print(f'A à F : EQM constante = {np.mean((y - y_moy) ** 2):.2f}, EQM droite = {eqm(a_mc, b_mc):.2f}, '
      f'R² = {r2_lecon(y, a_mc * x + b_mc, y_moy):.3f}, r² = {np.corrcoef(x, y)[0, 1] ** 2:.3f}')

# Les trois droites, la même mesure, sur A à F puis sur V1 à V4 (leçon 3.6)
droites = {'moindres carrés': (a_mc, b_mc), 'ouverture': (3, 30), 'collègue': (3.3, 14)}
for nom, (a, b) in droites.items():
    e_af, e_v = y - (a * x + b), yv - (a * xv + b)
    print(f'{nom:16s} A à F : EQM = {np.mean(e_af ** 2):6.2f}  EAM = {np.mean(np.abs(e_af)):5.2f}   |   '
          f'V1 à V4 : EQM = {np.mean(e_v ** 2):6.2f}  EAM = {np.mean(np.abs(e_v)):5.2f}   écarts {e_v.round(2)}')
yv_bis = np.array([132, 215, 254, 312.0])   # si V4 s'était vendu 312 k€
for nom in ('moindres carrés', 'collègue'):
    a, b = droites[nom]
    print(f'V4 à 312 k€ : {nom} EQM = {np.mean((yv_bis - (a * xv + b)) ** 2):.2f}')
print(f'G, exemple de calcul : {a_mc * x_G + b_mc:.2f} k€ prédits, écart {y_G - (a_mc * x_G + b_mc):.2f} k€')

## Exercice 3.1 — Lire un graphique : deux mesures, deux minimums

*Données D0.1. Notions : EAM et EQM (leçon 3.1), courbe de perte (leçon 3.2), médiane (leçon 1.3).* La pente est fixée à 3 k€/m². Répondez d'abord aux questions a) à e) en lisant la figure du livre.

### a) à e) Corrigé

**a)** L'EQM est la plus petite vers $b \approx 30$ k€ et vaut alors environ 57 k€². Le calcul de f) donne exactement $b = 30{,}33$ et 57,22 k€².

**b)** La courbe de l'EAM est faite de segments de droite. Autour de son minimum, elle forme un **palier** horizontal, de $b = 24$ à $b = 35$ k€, à la hauteur d'environ 7,3 k€. Toutes ces valeurs de $b$ donnent la même EAM.

**c)** Pour $b = 30$, on lit une EQM d'environ 57 k€² et une EAM d'environ 7,3 k€. Le tableau des écarts de la leçon 3.1 donne 57,33 k€² et 7,33 k€ pour la droite de l'ouverture : la lecture est cohérente.

**d)** Avec $b = 25$, les écarts valent $r_i - 25$ : −3, 12, −2, 10, −1 et 16 k€. Leurs valeurs absolues font 44 au total, comme pour $b = 30$ : l'EAM vaut encore 44 / 6 ≈ 7,33 k€. Selon l'EAM, les deux droites sont **équivalentes**. Leurs carrés font 9 + 144 + 4 + 100 + 1 + 256 = 514, d'où une EQM de 514 / 6 ≈ 85,67 k€², contre 57,33 : selon l'EQM, la droite du collègue est **moins bonne**. Le verdict dépend de la mesure, comme à la leçon 3.1.

**e)** Sur le palier, la pente de la courbe vaut 0. Une descente de gradient qui y arrive s'arrête là où elle se trouve : elle ne peut pas choisir entre 24 et 35 k€. Aux angles, la pente change brusquement et la dérivée n'existe pas. La courbe de l'EQM est une parabole lisse : elle a un seul point le plus bas, et sa pente, proportionnelle à la distance à ce point, indique à chaque pas dans quel sens aller et de combien.

### f) Dans le notebook

In [ ]:
def eqm_b(b):
    return np.mean((y - (3 * x + b)) ** 2)

def eam_b(b):
    return np.mean(np.abs(y - (3 * x + b)))

In [ ]:
# Vérification, puis tracé des deux courbes
if eqm_b(30) is None or eam_b(30) is None:
    print('Complétez eqm_b et eam_b.')
else:
    print('eqm_b et eam_b :', 'correct' if np.isclose(eqm_b(30), 344 / 6) and np.isclose(eam_b(30), 44 / 6) else 'à revoir')
    B = np.linspace(15, 45, 601)
    fig, (h, bas) = plt.subplots(2, 1, figsize=(6, 4.5), sharex=True)
    h.plot(B, [eqm_b(b) for b in B], color=BLEU); h.set_ylabel('EQM (k€²)'); h.grid(alpha=0.4)
    bas.plot(B, [eam_b(b) for b in B], color=ORANGE); bas.set_ylabel('EAM (k€)'); bas.grid(alpha=0.4)
    bas.set_xlabel('Ordonnée à l\'origine b (k€)')
    plt.show()
    print('angles de la courbe de l\'EAM (r_i rangés) :', np.sort(y - 3 * x))

In [ ]:
r_tries = np.sort(y - 3 * x)                    # 22, 23, 24, 35, 37, 41
mediane_r = (r_tries[2] + r_tries[3]) / 2       # n = 6 pair : rangs 3 et 4 → 29,5
moyenne_r = np.mean(y - 3 * x)                  # 182 / 6 ≈ 30,33

In [ ]:
# Vérification
for nom, la_votre, vrai in [('médiane', mediane_r, 29.5), ('moyenne', moyenne_r, 182 / 6)]:
    print(f'{nom} des r :', 'pas encore complété' if la_votre is None else ('correct' if np.isclose(la_votre, vrai) else 'à revoir'))
if mediane_r is not None and eam_b(30) is not None:
    print(f'EAM à la médiane : {eam_b(mediane_r):.2f} k€ (sur le palier) ; '
          f'EQM à la moyenne : {eqm_b(moyenne_r):.2f} k€² (minimum)')

### Corrigé de f)

Les angles se trouvent en 22, 23, 24, 35, 37 et 41 k€, les six valeurs $r_i$. Le palier va de la troisième à la quatrième valeur rangée. La médiane des $r_i$, $(24 + 35)/2 = 29{,}5$ k€, s'y trouve. Le minimum de l'EQM est atteint pour la moyenne des $r_i$, 30,33 k€. On retrouve le chapitre 1 : la moyenne répond à l'EQM et la médiane à l'EAM. La médiane résiste aux valeurs extrêmes, la moyenne non. De même, l'EQM pénalise davantage les grosses erreurs que l'EAM.

## Exercice 3.2 — Deux pas de descente de gradient

*Données D0.1. Notions : erreur quadratique moyenne (leçon 3.1), dérivée et mise à jour (leçon 3.3).* La pente est fixée à 3 k€/m². On cherche l'ordonnée à l'origine $b$. Faites d'abord les questions a) à d) sur papier.

### a) Préparer

In [ ]:
r = y - 3 * x
r

In [ ]:
# Vérification
if r is None:
    print('Complétez la cellule précédente.')
else:
    print('r :', 'correct' if np.allclose(r, [22, 37, 23, 35, 24, 41]) else 'à revoir')

### a) à c) Corrigé sur papier

**a)** $r = (22 ; 37 ; 23 ; 35 ; 24 ; 41)$ k€. Par exemple pour A : 118 − 3 × 32 = 118 − 96 = 22. L'écart vaut alors $e_i = y_i - 3x_i - b = r_i - b$.

**b) et c)** Avec $\eta = 0{,}3$, la mise à jour vaut $b - 0{,}3 \times \left(-\frac{2}{6}\sum e_i\right) = b + 0{,}1 \sum e_i$.

| Étape | $b$ (k€) | Écarts $r_i - b$ (k€) | $\sum e_i$ | $L(b)$ (k€²) | $L'(b)$ | Pas |
|:--:|--:|:--|--:|--:|--:|--:|
| 0 | 20 | 2 ; 17 ; 3 ; 15 ; 4 ; 21 | 62 | 984 / 6 = 164 | −20,67 | +6,2 |
| 1 | 26,2 | −4,2 ; 10,8 ; −3,2 ; 8,8 ; −2,2 ; 14,8 | 24,8 | 445,84 / 6 ≈ 74,31 | −8,27 | +2,48 |
| 2 | 28,68 | −6,68 ; 8,32 ; −5,68 ; 6,32 ; −4,68 ; 12,32 | 9,92 | ≈ 59,96 | −3,31 | |

Détail de l'étape 0 : les carrés valent 4, 289, 9, 225, 16 et 441, pour une somme de 984. La dérivée vaut $-\frac{2}{6} \times 62 \approx -20{,}67$. Le pas vaut $0{,}3 \times 20{,}67 = 6{,}2$, et $b_1 = 20 + 6{,}2 = 26{,}2$.

In [ ]:
L_b0 = 164
derivee_b0 = -20.67
b1 = 26.2
L_b1 = 74.31
b2 = 28.68
L_b2 = 59.96

In [ ]:
# Vérification (calcul indépendant)
def _L(b): return np.mean((y - 3 * x - b) ** 2)
def _d(b): return -2 / 6 * np.sum(y - 3 * x - b)
_b1 = 20 - 0.3 * _d(20); _b2 = _b1 - 0.3 * _d(_b1)
attendus = {'L_b0': (L_b0, _L(20)), 'derivee_b0': (derivee_b0, _d(20)), 'b1': (b1, _b1),
            'L_b1': (L_b1, _L(_b1)), 'b2': (b2, _b2), 'L_b2': (L_b2, _L(_b2))}
for nom, (la_votre, vrai) in attendus.items():
    etat = 'pas encore complété' if la_votre is None else ('correct' if abs(la_votre - vrai) < 0.01 else 'à revoir')
    print(f'{nom} : {etat}')

### d) Interpréter

La perte diminue à chaque pas : 164, puis 74,31, puis 59,96 k€². Les pas raccourcissent : 6,2, puis 2,48. Avec $\eta = 0{,}3$, chaque pas réduit de 60 % la distance au minimum. En effet, $b_{\text{nouveau}} = b + 0{,}1 \times 6\,(\bar{r} - b)$, donc $\bar{r} - b$ est multiplié par 0,4. La descente se dirige vers $\bar{r} = 182/6 \approx 30{,}33$ k€, la **moyenne des $r_i$**. Au minimum, la perte vaut la variance des $r_i$, environ 57,22 k€², soit $7{,}56^2$ : c'est l'écart-type des écarts de la droite de l'ouverture (exercice 1.6).

### e) Dans le notebook

In [ ]:
def perte(b):
    """EQM de la droite y = 3x + b sur A à F (équation 3.2)."""
    e = y - (3 * x + b)
    return np.mean(e ** 2)

def derivee(b):
    """L'(b) = −(2/n) Σ e_i (équation 3.6)."""
    e = y - (3 * x + b)
    return -2 / len(x) * np.sum(e)

Les deux fonctions reprennent mot pour mot les équations 3.2 et 3.6, avec une pente fixée à 3.

In [ ]:
def descente(b0, eta, nb_pas):
    """Renvoie la liste des b et la liste des pertes, de l'étape 0 à l'étape nb_pas."""
    bs, pertes = [b0], [perte(b0)]
    for _ in range(nb_pas):
        bs.append(bs[-1] - eta * derivee(bs[-1]))   # équation 3.5
        pertes.append(perte(bs[-1]))
    return bs, pertes

if perte(20) is None or derivee(20) is None:
    print('Complétez perte et derivee.')
else:
    bs, pertes = descente(20, 0.3, 2)
    print('deux pas :', np.round(bs, 3), np.round(pertes, 2), '→',
          'correct' if np.allclose(bs, [20, 26.2, 28.68]) else 'à revoir')
    bs, pertes = descente(20, 0.3, 20)
    plt.figure(figsize=(6, 2.8))
    plt.plot(pertes, marker='o', color=BLEU)
    plt.xlabel('Pas'); plt.ylabel('Perte (k€²)'); plt.grid(alpha=0.4)
    plt.show()
    print('b après 20 pas :', round(bs[-1], 3), '  moyenne des r :', round(np.mean(y - 3 * x), 3))

### f) Prédire, puis vérifier : $\eta = 1{,}2$

**Prédiction.** Avec $\eta = 1{,}2$, la mise à jour multiplie la distance au minimum par $1 - 2\eta = -1{,}4$. Chaque pas franchit le minimum et atterrit plus loin de l'autre côté : $b$ oscille de part et d'autre de 30,33 en s'en éloignant, et la perte augmente. La descente **diverge** : 20 → 44,8 → 10,08 → 58,69 → −9,36… Un taux trop grand ne ralentit pas la descente : il la fait échouer.

In [ ]:
if perte(20) is not None and derivee(20) is not None:
    fig, axes = plt.subplots(1, 2, figsize=(8, 2.8))
    for ax, eta in zip(axes, (0.3, 1.2)):
        bs, pertes = descente(20, eta, 6)
        ax.plot(bs, marker='o', color=ORANGE if eta > 1 else BLEU)
        ax.axhline(np.mean(y - 3 * x), color='black', linewidth=0.8, linestyle='--')
        ax.set_title(f'η = {eta}', fontsize=10); ax.set_xlabel('Pas'); ax.set_ylabel('b (k€)')
        print(f'η = {eta} :', np.round(bs, 2))
    plt.tight_layout(); plt.show()

### g) Exemple de calcul sur G

In [ ]:
prix_G = 3 * x_G + 28.68     # = 186 + 28,68 = 214,68 k€
ecart_G = y_G - prix_G       # = 209 − 214,68 = −5,68 k€

In [ ]:
# Vérification
if prix_G is None or ecart_G is None:
    print('Complétez la cellule précédente.')
else:
    print('G :', 'correct' if np.isclose(prix_G, 214.68) and np.isclose(ecart_G, -5.68) else 'à revoir')

### Corrigé de g)

Prix prédit : 3 × 62 + 28,68 = 186 + 28,68 = **214,68 k€**. Écart signé : 209 − 214,68 = **−5,68 k€**. G n'a pas servi à ajuster $b$. Ce calcul montre comment utiliser la règle obtenue après deux pas, mais il ne mesure pas sa qualité sur des ventes nouvelles : un seul logement ne le permettrait pas, et le chapitre 6 traitera cette question.

## Exercice 3.3 — Que veut dire « 99 % » ?

*Données D0.1 et D3.1. Notions : coefficient de détermination, ajustement et évaluation (leçon 3.6).* Faites d'abord les questions a) à f) sur papier.

### a) à f) Corrigé

**a)** Avec l'équation 3.10 et les EQM de la leçon 3.1 : droite de l'ouverture, $R^2 = 1 - 57{,}33 / 5\,205{,}22 \approx 0{,}989$ ; droite du collègue, $R^2 = 1 - 65{,}11 / 5\,205{,}22 \approx 0{,}987$. La référence est la prédiction constante égale au prix moyen de A à F, 222,33 k€, et les deux EQM sont calculées sur A à F.

**b)** Le dénominateur, l'EQM de la prédiction constante sur A à F, est le même pour toutes les droites. Le $R^2$ est donc d'autant plus grand que l'EQM est petite. Or la droite des moindres carrés est, par définition, celle dont l'EQM sur A à F est la plus petite. Aucune droite ne peut avoir un meilleur $R^2$ sur ces six ventes.

**c)** Prix prédits : 3 × 38 + 30 = 144, 3 × 58 + 30 = 204, 3 × 76 + 30 = 258 et 3 × 92 + 30 = 306 k€. Écarts : 132 − 144 = −12, 215 − 204 = +11, 254 − 258 = −4 et 322 − 306 = +16 k€. Carrés : 144, 121, 16 et 256, pour une somme de 537. EQM : 537 / 4 = 134,25 k€².

**d)** Sur A à F : moindres carrés (45,36), ouverture (57,33), collègue (65,11). Sur V1 à V4 : collègue (70,73), moindres carrés (93,29), ouverture (134,25). Le renversement ne prouve pas que la droite du collègue soit meilleure. Les trois droites sont proches et quatre ventes, c'est très peu : avec V4 vendu 312 k€ au lieu de 322, la droite des moindres carrés repasserait devant (61,11 contre 73,73 k€²). Il montre seulement que le classement obtenu pendant l'ajustement ne se transporte pas tel quel sur des ventes nouvelles.

**e)** Par exemple : « Sur les six ventes qui nous ont servi à tracer notre droite, elle réduit de 98,9 % l'erreur quadratique moyenne de la prédiction constante égale à leur prix moyen, 222,33 k€. Sur quatre ventes nouvelles, son erreur quadratique moyenne vaut 134,25 k€², soit environ 11,6 k€. » Le mot « explique » est à éviter : le $R^2$ compare deux erreurs, il ne mesure pas une cause.

**f)** C'est refaire sur V1 à V4 ce que l'ajustement faisait sur A à F. Dès qu'on se sert de ces ventes pour choisir, elles ne sont plus nouvelles pour la droite retenue : son EQM sur V1 à V4 devient à son tour une erreur de sélection, optimiste. Pour annoncer une erreur honnête, il faudrait d'autres ventes, jamais utilisées ni pour ajuster ni pour choisir. Le chapitre 6 organisera ce découpage.

### g) Dans le notebook

In [ ]:
def eqm_ventes(observes, predits):
    """observes et predits : tableaux numpy de même longueur."""
    return np.mean((observes - predits) ** 2)

In [ ]:
# Vérification : le classement de la leçon 3.6, puis le R² sur A à F (référence : prix moyen de A à F)
if eqm_ventes(y, y) is None:
    print('Complétez la fonction eqm_ventes.')
else:
    a_mc = np.mean((x - x.mean()) * (y - y.mean())) / x.var()
    b_mc = y.mean() - a_mc * x.mean()
    attendus = {'moindres carrés': ((a_mc, b_mc), 45.36, 93.29), 'ouverture': ((3, 30), 57.33, 134.25),
                'collègue': ((3.3, 14), 65.11, 70.73)}
    for nom, ((a, b), att_af, att_v) in attendus.items():
        e1, e2 = eqm_ventes(y, a * x + b), eqm_ventes(yv, a * xv + b)
        print(f'{nom:16s} A à F : {e1:6.2f}   V1 à V4 : {e2:6.2f} →',
              'correct' if abs(e1 - att_af) < 0.006 and abs(e2 - att_v) < 0.006 else 'à revoir')
    r2_af = 1 - eqm_ventes(y, a_mc * x + b_mc) / eqm_ventes(y, np.full(6, y.mean()))
    print(f'R² sur A à F = {r2_af:.4f}, r² = {np.corrcoef(x, y)[0, 1] ** 2:.4f} →',
          'correct' if np.isclose(r2_af, np.corrcoef(x, y)[0, 1] ** 2) else 'à revoir')

## Exercice 3.4 — Une descente sur surfaces standardisées

*Données D0.1. Notions : gradient (leçon 3.4), standardisation et reconversion (leçon 3.5), droite des moindres carrés (leçon 3.6).*

### a) Corrigé

Avec $(w ; c) = (56{,}99 ; 190)$ : $56{,}99 \times 1{,}4915 + 190 \approx 85{,}00 + 190 = 275{,}0$ k€. Avec $(a ; b) = (2{,}5 ; 30)$ : $2{,}5 \times 98 + 30 = 245 + 30 = 275$ k€. Les deux écritures donnent la même prédiction pour F, comme l'annonce l'équation 2.7.

### b) Moyenne, écart-type et scores standardisés

In [ ]:
mu = x.mean()            # 64 m²
sigma = x.std()          # 22,796 m² : np.std divise par n par défaut
z = (x - mu) / sigma
z.round(4)

In [ ]:
# Vérification
if z is None:
    print('Complétez la cellule précédente.')
else:
    print('scores z :', 'correct' if np.isclose(np.mean(z), 0) and np.isclose(np.mean(z ** 2), 1) and np.isclose(mu, 64) else 'à revoir')

### c) Le gradient dans l'écriture standardisée

In [ ]:
def gradient_std(w, c):
    e = y - (w * z + c)
    return np.array([-2 / len(x) * np.sum(z * e), -2 / len(x) * np.sum(e)])

In [ ]:
# Vérification au point de départ (mêmes prédictions que (2,5 ; 30))
if z is None or gradient_std(1, 1) is None:
    print('Complétez les cellules b) et c).')
else:
    w0, c0 = 2.5 * sigma, 30 + 2.5 * mu
    g0 = gradient_std(w0, c0)
    print(f'départ : w0 = {w0:.2f}, c0 = {c0:.0f}, gradient = {g0.round(2)} →',
          'correct' if np.allclose(g0, [-29.683, -64.667], atol=0.001) else 'à revoir')

### d) Trente pas, puis retour aux unités d'origine

In [ ]:
w, c = 2.5 * sigma, 30 + 2.5 * mu
for _ in range(30):
    w, c = np.array([w, c]) - 0.3 * gradient_std(w, c)
a_std = w / sigma
b_std = c - a_std * mu
print(f'w = {w:.3f}, c = {c:.3f}  →  a = {a_std:.3f} k€/m², b = {b_std:.2f} k€')

In [ ]:
# Vérification : équation 3.9 et np.polyfit
a_mc = np.mean((x - x.mean()) * (y - y.mean())) / x.var()
b_mc = y.mean() - a_mc * x.mean()
print(f'équation 3.9 : a = {a_mc:.3f}, b = {b_mc:.2f} ; np.polyfit : {np.round(np.polyfit(x, y, 1), 3)}')
if a_std is None:
    print('Complétez la cellule précédente.')
else:
    print('droite obtenue :', 'correct' if np.allclose([a_std, b_std], [a_mc, b_mc], atol=1e-4) else 'à revoir')

### e) et f) La perte des deux descentes, puis le nombre de pas

In [ ]:
# Les deux descentes de la leçon 3.5, pas par pas (rien à compléter)
def trajet_m2(nb_pas, eta=0.00006):
    a, b, pertes = 2.5, 30.0, []
    for _ in range(nb_pas + 1):
        pertes.append(np.mean((y - a * x - b) ** 2))
        e = y - a * x - b
        a, b = a + eta * 2 / 6 * np.sum(x * e), b + eta * 2 / 6 * np.sum(e)
    return pertes

def trajet_std(nb_pas, eta=0.3):
    zz = (x - x.mean()) / x.std()
    w, c, pertes = 2.5 * x.std(), 30 + 2.5 * x.mean(), []
    for _ in range(nb_pas + 1):
        pertes.append(np.mean((y - w * zz - c) ** 2))
        e = y - w * zz - c
        w, c = w + eta * 2 / 6 * np.sum(zz * e), c + eta * 2 / 6 * np.sum(e)
    return pertes

plt.figure(figsize=(6, 3))
plt.semilogy(trajet_m2(30), marker='o', color=ORANGE, label='surfaces en m², η = 0,000 06')
plt.semilogy(trajet_std(30), marker='s', color=BLEU, label='surfaces standardisées, η = 0,3')
plt.axhline(45.364, color='black', linestyle='--', linewidth=0.8)
plt.xlabel('Nombre de pas'); plt.ylabel('Perte (k€²)'); plt.legend(); plt.grid(alpha=0.4); plt.show()

In [ ]:
# f) Nombre de pas pour arriver à moins de 0,1 k€² du minimum (quelques secondes)
L_min = np.mean((y - (a_mc * x + b_mc)) ** 2)

def nb_pas(X, a, b, eta, limite=10 ** 6):
    for k in range(1, limite + 1):
        e = y - a * X - b
        a, b = a + eta * 2 / 6 * np.sum(X * e), b + eta * 2 / 6 * np.sum(e)
        if np.mean((y - a * X - b) ** 2) - L_min < 0.1:
            return k

pertes_std = trajet_std(30)
print('surfaces standardisées, η = 0,3 :', next(k for k, v in enumerate(pertes_std) if v - L_min < 0.1), 'pas')
print('surfaces en m², η = 0,000 06    :', nb_pas(x, 2.5, 30.0, 0.00006), 'pas')

### Corrigé de f)

La descente standardisée arrive à moins de 0,1 k€² du minimum en **6 pas**, la descente en m² en **169 808 pas**. Les deux trouvent la même droite, 3,151 × surface + 20,67. À retenir : avant une descente de gradient, on standardise les colonnes d'échelles très différentes. Pour lire les coefficients dans les unités d'origine, on les reconvertit ensuite avec l'équation 3.8.

## Exercice 3.5 — Prédire, puis vérifier : des surfaces en dizaines de m²

*Données D0.1. Notions : changement d'échelle (leçon 2.5), gradient (leçon 3.4), standardisation (leçon 3.5).*

### a) à d) Prédictions attendues

**a)** La droite ne change pas : ce sont les mêmes logements et les mêmes prix, seule l'unité de la colonne change. Une dizaine de m² vaut 10 m², donc la pente est multipliée par 10 : 31,51 k€ par dizaine de m². L'ordonnée à l'origine, en k€, reste 20,67.

**b)** Le réglage de départ devient (25 ; 30). Les écarts sont les mêmes, et chaque $x_i$ est divisé par 10. Le quotient vaut donc 74,5 / 10 ≈ 7,45 dizaines de m², c'est-à-dire toujours 74,5 m² : la même surface, exprimée dans une autre unité.

**c)** Plus grand. La dérivée par rapport à la pente est divisée par 10, et un même déplacement de la droite demande une pente 10 fois plus grande. Le taux limite est donc multiplié par environ 10 × 10 = 100 : il passe d'environ 0,000 22 à environ 0,021.

**d)** Moins de pas que la descente en m², car les pas peuvent être 100 fois plus grands. Mais plus que la descente standardisée : les surfaces ne sont pas centrées, et la vallée reste inclinée.

### e) Vérifier

In [ ]:
x_diz = x / 10
x_diz

In [ ]:
# Vérification, puis les mesures (rien d'autre à compléter)
if x_diz is None:
    print('Complétez la cellule précédente.')
else:
    print('x_diz :', 'correct' if np.allclose(x_diz, [3.2, 4.5, 5.4, 7.0, 8.5, 9.8]) else 'à revoir')
    print('a) droite des moindres carrés (pente, ordonnée) :', np.round(np.polyfit(x_diz, y, 1), 2))
    e0 = y - (25 * x_diz + 30)
    print(f'b) quotient à l\'étape 0 : {np.sum(x_diz * e0) / np.sum(e0):.2f} dizaines de m²')
    for eta in (0.021, 0.022):
        a, b = 25.0, 30.0
        for _ in range(200):
            e = y - a * x_diz - b
            a, b = a + eta * 2 / 6 * np.sum(x_diz * e), b + eta * 2 / 6 * np.sum(e)
        print(f'c) η = {eta} : perte après 200 pas = {np.mean((y - a * x_diz - b) ** 2):.4g} k€²')
    L_min = np.mean((y - np.polyval(np.polyfit(x, y, 1), x)) ** 2)
    a, b = 25.0, 30.0
    for k in range(1, 10 ** 6):
        e = y - a * x_diz - b
        a, b = a + 0.006 * 2 / 6 * np.sum(x_diz * e), b + 0.006 * 2 / 6 * np.sum(e)
        if np.mean((y - a * x_diz - b) ** 2) - L_min < 0.1:
            break
    print(f'd) η = 0,006 : {k} pas pour arriver à moins de 0,1 k€² du minimum')

### e) et f) Corrigé

Le notebook confirme les prédictions : pente 31,51 k€ par dizaine de m², ordonnée 20,67 k€ ; quotient 7,45 ; avec 0,021, la descente converge, avec 0,022, elle diverge ; avec $\eta = 0{,}006$, il faut **1 785 pas**, contre 169 808 pour les surfaces en m² et 6 pour les surfaces standardisées (exercice 3.4).

Changer d'unité n'a pas changé la droite optimale : 31,51 k€ par dizaine de m², c'est 3,151 k€ par m². Cela a changé le trajet et le taux utilisable. La standardisation va plus loin. Elle divise par l'écart-type, ce qui met les deux coefficients à la même échelle. Elle retire aussi la moyenne : avec $\sum z_i = 0$, la pente et l'ordonnée à l'origine ne se gênent plus, et la vallée cesse d'être inclinée (exercice 3.6).

## Exercice 3.6 — Défi : des cercles sur la carte

*Données D0.1. Notions : standardisation (leçons 1.4 et 3.5), dérivées partielles et gradient (leçon 3.4).* Les questions a) à f) se font sur papier. Le notebook vérifie vos résultats.

### a) à f) Corrigé

**a)** $\sum z_i = \sum (x_i - \mu)/\sigma = (\sum x_i - n\mu)/\sigma = 0$, puisque $n\mu = \sum x_i$. Et $\sum z_i^2 = \sum (x_i - \mu)^2 / \sigma^2 = n\sigma^2 / \sigma^2 = n$, puisque l'écart-type divise par $n$.

**b)** $\partial L / \partial c = -\frac{2}{n} \sum (y_i - w z_i - c) = -\frac{2}{n}(n\bar{y} - w \times 0 - nc) = 2(c - \bar{y})$. $\partial L / \partial w = -\frac{2}{n} \sum z_i (y_i - w z_i - c) = -\frac{2}{n}(nm - wn - c \times 0) = 2(w - m)$. Les deux dérivées s'annulent pour $w^* = m$ et $c^* = \bar{y} = 222{,}33$ k€. Ici $m = 1\,637{,}5 / 22{,}796 \approx 71{,}83$ k€. Reconverti avec l'équation 3.8 : $a = m / \sigma = \operatorname{cov}(x, y)/\sigma_x^2 \approx 3{,}151$, la pente de l'équation 3.9.

**c)** $w_{\text{nouveau}} = w - \eta \times 2(w - w^*)$, donc $w_{\text{nouveau}} - w^* = (1 - 2\eta)(w - w^*)$. Le calcul est le même pour $c$. Avec $\eta = 0{,}3$, le facteur vaut $1 - 0{,}6 = 0{,}4$. Dans l'exercice 3.2, la dérivée valait déjà $2(b - \bar{r})$, d'où le même facteur.

**d)** Avec $\eta = 0{,}5$, le facteur vaut 0 : un seul pas amène au minimum. La descente converge tant que $|1 - 2\eta| < 1$, c'est-à-dire $0 < \eta < 1$. Avec $\eta = 1$, le facteur vaut −1 : le réglage rebondit indéfiniment entre deux points. Au-delà de 1, la descente diverge.

**e)** En développant le carré et en utilisant a) : $L = \overline{y^2} + w^2 + c^2 - 2wm - 2c\bar{y} = (w - m)^2 + (c - \bar{y})^2 + \overline{y^2} - m^2 - \bar{y}^2$, où $\overline{y^2}$ est la moyenne des carrés des prix. Le dernier terme est la valeur de $L$ au minimum : $54\,637{,}33 - 5\,159{,}86 - 49\,432{,}11 \approx 45{,}36$ k€². Une courbe de niveau réunit les réglages de même perte, donc de même distance $(w - w^*)^2 + (c - c^*)^2$ au minimum : c'est un cercle centré sur le minimum. Le gradient pointe vers le centre, d'où le trajet presque rectiligne de la leçon 3.5.

**f)** En m², $\sum x_i = 384 \neq 0$ et $\frac{1}{n}\sum x_i^2 \approx 4\,615{,}67 \neq 1$. Le développement de $L(a, b)$ contient alors un terme croisé $2 \times 64\, ab$, et $a^2$ y pèse environ 4 600 fois plus que $b^2$. Les courbes de niveau sont des ellipses très allongées et inclinées : c'est la vallée de la figure du livre.

### Vérifier dans le notebook

In [ ]:
zz = (x - x.mean()) / x.std()
print(f'a) somme des z = {np.sum(zz):.1e}, somme des z² = {np.sum(zz ** 2):.4f}')

In [ ]:
m = np.mean(zz * y)

def un_pas(w, c, eta):
    """Renvoie (w, c) après une mise à jour (équation 3.7 avec z à la place de x)."""
    e = y - (w * zz + c)
    return w + eta * 2 / len(x) * np.sum(zz * e), c + eta * 2 / len(x) * np.sum(e)

In [ ]:
# Vérification de b) et d)
if m is None or un_pas(1, 1, 0.1) is None:
    print('Complétez la cellule précédente.')
else:
    print(f'b) m = {m:.2f} →', 'correct' if np.isclose(m, np.mean((x - x.mean()) * (y - y.mean())) / x.std()) else 'à revoir')
    w0, c0 = 2.5 * x.std(), 30 + 2.5 * x.mean()
    w1, c1 = un_pas(w0, c0, 0.5)
    print(f'd) un pas avec η = 0,5 : (w ; c) = ({w1:.2f} ; {c1:.2f}) →',
          'correct' if np.allclose([w1, c1], [m, y.mean()]) else 'à revoir')
    for eta in (1.0, 1.05):
        w, c = w0, c0
        suite = []
        for _ in range(4):
            w, c = un_pas(w, c, eta)
            suite.append(round(c, 2))
        print(f'   η = {eta} : c après 1, 2, 3, 4 pas = {suite}')
    W, Cc = np.meshgrid(np.linspace(40, 100, 200), np.linspace(180, 260, 200))
    Z = np.mean([(yi - W * zi - Cc) ** 2 for zi, yi in zip(zz, y)], axis=0)
    plt.figure(figsize=(4.5, 4.5))
    plt.contour(W, Cc, Z, levels=[50, 100, 200, 400, 800, 1600], colors=BLEU)
    plt.gca().set_aspect('equal')
    plt.xlabel('w (k€ par écart-type)'); plt.ylabel('c (k€)'); plt.title('e) Des cercles'); plt.show()